In [4]:
import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report


In [7]:
print("Downloding fashion MNIST dataset(this may take 30-60 seconds)..")
fashion_mnist =fetch_openml('Fashion-MNIST',version=1,as_frame = False)
x= fashion_mnist.data
y = fashion_mnist.target.astype(int)
print(f"total dataset size : {x.shape[0]} images ,each with {x.shape[1]} pixels.")

Downloding fashion MNIST dataset(this may take 30-60 seconds)..
total dataset size : 70000 images ,each with 784 pixels.


In [8]:
x_subset,_,y_subset,_ = train_test_split(
    x,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

In [9]:
x_train,x_test,y_train,y_test = train_test_split(
    x_subset,y_subset,
    train_size=2000,
    stratify=y_subset,
    random_state=42
)
print(f"Training images:{x_train.shape[0]}")
print(f"Testing images:{x_test.shape[0]}")

Training images:2000
Testing images:10000


In [10]:
print(f"Before scaling -> Min: {x_train.min()},Max:{x_train.max()}")
x_train = x_train / 255.0
x_test = x_test / 255.0
print(f"After scaling -> Min :{x_train.min()},Max:{x_train.max()}")

Before scaling -> Min: 0,Max:255
After scaling -> Min :0.0,Max:1.0


In [16]:
k_values = [1, 3, 5, 7, 9, 15]
result = {}

print(f"{'K Value':<8}|{'Accuracy':<10}|{'Prediction Time(seconds)':<25}")
print("-" * 50)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    knn.fit(x_train, y_train)

    start_time = time.time()
    y_pred = knn.predict(x_test)
    elapsed_time = time.time() - start_time

    acc = accuracy_score(y_test, y_pred)

    result[k] = {
        "accuracy": acc,
        "time": elapsed_time,
        "predictions": y_pred
    }

    print(f"{k:<8}|{acc*100:.2f}%|{elapsed_time:<25.2f}")


K Value |Accuracy  |Prediction Time(seconds) 
--------------------------------------------------
1       |76.81%|3.09                     
3       |77.19%|0.14                     
5       |77.71%|0.13                     
7       |77.29%|0.13                     
9       |77.29%|0.14                     
15      |76.55%|0.14                     


In [19]:
class_names=[
    "T-shirt/top","trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]
best_k = max(result,key=lambda k:result[k]["accuracy"])
print(f"Best K is :{best_k} with {result[best_k]["accuracy"]*100:2f}%accuracy\n")
print("Per-class Classification Report:")
print(classification_report(y_test,result[best_k]["predictions"],target_names =class_names))

Best K is :5 with 77.710000%accuracy

Per-class Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.71      0.83      0.77      1000
     trouser       0.97      0.95      0.96      1000
    Pullover       0.60      0.69      0.64      1000
       Dress       0.80      0.84      0.82      1000
        Coat       0.65      0.56      0.60      1000
      Sandal       0.97      0.70      0.82      1000
       Shirt       0.52      0.45      0.48      1000
     Sneaker       0.79      0.91      0.84      1000
         Bag       0.97      0.90      0.93      1000
  Ankle boot       0.85      0.95      0.90      1000

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.77     10000
weighted avg       0.78      0.78      0.77     10000

